# kpi_eye_accuracy.ipynb — KPI ① 눈 감김 판정 정확도 (A, P0)

- 팀원 사진 · 영상 프레임 (팀원 a · b · c · d · 팀장 t × open · half · closed, 영상은 밝은 곳 · 어두운 곳) — **시연 폰 전면 카메라, 거치대 위치**
- 영상 프레임은 `tools/kpi_frames.py` 가 **앱 카메라 크기(640×480, 4:3)** 로 맞춘 것 (원본 크기는 `images/_full/`)
- 비교: **ML Kit 눈 뜸 확률** · **폰 EAR**(ML Kit 눈 윤곽) · **서버 EAR**(MediaPipe, 고정 / 개인 기준) · **CNN**(MRL만 / AI-Hub 미세조정)
- 결과로 기준값 확정 → `sleep_detector_app/lib/config.dart` · `sleep_detector_api/config.py`, **서버 확인 방식(CNN / EAR) 결정** (3주차 수요일)
- **'졸았는데 못 잡은 비율'을 따로 표시** — 졸음 감지는 놓치는 쪽이 더 위험
- 🔴 측정용 사진은 CNN 학습에 쓰지 않음. 팀 사진 · CSV 는 **Google Drive 에만** (저장소에 올리지 않음)
- 🔴 GitHub 에 올리기 전: **모든 출력 지우기** (사진 이름 · 숫자가 출력에 남지 않게)

### 입력 CSV (같은 사진으로)

| 파일 | 만드는 법 | 칸 |
|---|---|---|
| `eye_measure.csv` (폰) | 시연 폰 측정 도구 (`lib/tools/eye_measure_main.dart`) → 사진 선택 → `adb pull /sdcard/Android/data/com.example.sleep_detector_app/files/eye_measure.csv` | file, label, eye_open_prob, phone_ear, ms |
| `result_mrl.csv` (서버, CNN = MRL만) | `config.USE_CNN = True` · `CNN_MODEL_PATH = ml_models/eye_cnn.pt` → `python services/face_service.py images` → `result_images/result.csv` 이름 바꾸기 | file, label, face_found, left_ear, right_ear, ear, eye_closed_prob, method, eye_width_px, result |
| `result_aihub.csv` (서버, CNN = 미세조정) | 위와 같이, `CNN_MODEL_PATH = ml_models/eye_cnn_aihub.pt` | 〃 |

- 서버 CSV 는 **있는 것만** 읽는다. `result.csv` 하나만 있어도 된다 (CNN 없이 돌렸으면 CNN 비교는 건너뜀)
- 파일 이름 규칙 (`tools/kpi_frames.py` 가 붙임): **`사람_조명_[조건_]상태_출처번호.jpg`**
  - 예) `a_bright_closed_v03.jpg` (영상 프레임) · `t_bright_open_s02.jpg` (사진) · `b_bright_mask_open_s01.jpg` (조건 사진)
  - 사람 = 첫 덩어리, 조명 = `bright` / `dark`, 상태 = `open` / `half` / `closed`, 출처 = `_v번호` 영상 · `_s번호` 사진
  - **조건** = `mask`(마스크) · `sun`(선글라스) · `turn` / `down`(고개 돌림 · 숙임) → **기준값 계산에서 빼고 10번에서 따로** 집계

**실행 순서:** 0 → 10 차례대로. 숫자(기준값 · 목표)는 0번 셀에서만 바꾼다.

## 0. 준비 — CSV 위치 · 지금 기준값

In [ ]:
import os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

try:                                   # Colab 이면 드라이브에서 읽음
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/sleep_detector/kpi_eye'   # ← CSV 를 넣은 드라이브 폴더
except ImportError:                    # PC Jupyter 면 이 노트북 옆 폴더
    DATA_DIR = os.environ.get('KPI_EYE_DIR', 'kpi_eye')
PHONE_CSV = os.path.join(DATA_DIR, 'eye_measure.csv')
# 서버 CSV: {이름: 파일}. 있는 것만 읽음 — 첫 번째로 찾은 파일의 EAR · 눈 폭을 씀 (EAR 은 모델과 상관없이 같음)
SERVER_CSVS = {'CNN (MRL만)': 'result_mrl.csv', 'CNN (미세조정)': 'result_aihub.csv', 'CNN': 'result.csv'}
OUT_DIR = os.path.join(DATA_DIR, 'kpi_out')                      # 그림 · 요약표 저장
os.makedirs(OUT_DIR, exist_ok=True)

# 지금 기준값 (config.dart · config.py 와 같게 — 바꾸면 여기도 바꾼다)
APP_EYE_CLOSED_PROB = 0.4    # 앱: 두 눈 평균 뜸 확률 < 이 값 → 감김 (11/02 0.4 확정 — 개인 기준도 5명 모두 0.4)
APP_EAR_CLOSED = 0.2         # 앱: 폰 EAR < 이 값 → 감김 (useContourEar 켰을 때)
SERVER_EAR = 0.23            # 서버: EAR < 이 값 → 감음 (10/28 CNN 채택 회의 0.20 → 0.23)
CONFIRM_OPEN_MARGIN = 0.03   # 서버: EAR ≥ 기준 + 이 값 → '확실히 뜸' (앱 의심 취소)
CNN_CLOSED_PROB = 0.4        # 서버: CNN 감음 확률 ≥ 이 값 → 감음 (10/28 CNN 채택 회의: 미세조정 모델 · 0.4)
CNN_OPEN_MARGIN = 0.1        # 서버: CNN 감음 확률 < 기준 − 이 값 → '확실히 뜸' (취소)
CNN_PROB_CANDIDATES = [0.4, 0.5]   # 서버 확인 단계에서 비교할 CNN 기준 (0.4 = 10/28 확정, 0.5 = 예전 값)

# 개인 기준 (config 와 같게) — 사람마다 '뜸' 사진 CALIB_N 장으로 만들고, 그 사진은 평가에서 뺀다
CALIB_N = 5
APP_CALIB_RATIO, APP_CALIB_MIN, APP_CALIB_MAX = 0.5, 0.2, 0.4
SERVER_CALIB_RATIO, SERVER_CALIB_MIN, SERVER_CALIB_MAX = 0.7, 0.15, 0.25
PHONE_EAR_CALIB_RATIO, PHONE_EAR_CALIB_MIN, PHONE_EAR_CALIB_MAX = 0.7, 0.10, 0.30   # 폰 EAR 은 아직 config 에 없음 (제안값)

# 추천 기준값을 고르는 규칙: 놓친 비율이 이 값 이하인 기준 중 균형 정확도가 가장 높은 것
MAX_MISS = 0.05

# 조건 사진 (파일 이름에 있으면 기준값 계산에서 빼고 10번에서 따로)
COND_NAMES = {'mask': '마스크', 'sun': '선글라스', 'turn': '고개 돌림', 'down': '고개 숙임'}

## 1. 읽기 + 합치기

- 폰 CSV 와 서버 CSV 들을 `file` 로 합친다. 한쪽에만 있는 사진은 표로 보여 준다 (갤러리에서 고를 때 빠졌거나 이름이 바뀐 것)
- 파일 이름에서 사람 · 조명 · 상태 · 출처를 읽는다

In [ ]:
def read_csv(path):
    return pd.read_csv(path, encoding='utf-8-sig')

phone = read_csv(PHONE_CSV)
servers = {name: read_csv(os.path.join(DATA_DIR, f)) for name, f in SERVER_CSVS.items()
           if os.path.exists(os.path.join(DATA_DIR, f))}
if 'CNN' in servers and len(servers) > 1:      # 모델별 CSV 가 있으면 result.csv 는 안 씀
    servers.pop('CNN')
assert servers, f'서버 CSV 가 없습니다: {list(SERVER_CSVS.values())} 중 하나를 {DATA_DIR} 에 넣어 주세요'
print('폰', len(phone), '장 / 서버', {k: len(v) for k, v in servers.items()})

base_name, base = next(iter(servers.items()))
server = base.drop(columns=['label', 'eye_closed_prob', 'method'], errors='ignore')
CNN_COLS = {}                                   # 표시 이름 → 칸 이름
for i, (name, s) in enumerate(servers.items()):
    s = s.assign(eye_closed_prob=pd.to_numeric(s.get('eye_closed_prob'), errors='coerce'))
    if s['eye_closed_prob'].notna().any():
        col = f'cnn{i}'
        CNN_COLS[name] = col
        server = server.merge(s[['file', 'eye_closed_prob']].rename(columns={'eye_closed_prob': col}), on='file', how='left')

df = phone.merge(server, on='file', how='outer', indicator=True)
only = df[df['_merge'] != 'both'][['file', '_merge']]
if len(only):
    print('⚠️ 한쪽에만 있는 사진', len(only), '장 (left_only = 폰만, right_only = 서버만)')
    display(only)
df = df[df['_merge'] == 'both'].drop(columns='_merge').copy()

def parse(name):
    stem = os.path.splitext(name.lower())[0]
    tok = stem.split('_')
    m = re.search(r'_(v|s)\d+$', stem)
    return pd.Series({
        'person': tok[0],
        'light': next((t for t in tok if t in ('bright', 'dark')), ''),
        'label': next((l for l in ('closed', 'half', 'open') if l in tok or l in stem), ''),
        'cond': next((t for t in tok if t in COND_NAMES), ''),
        'source': {'v': '영상', 's': '사진'}.get(m.group(1), '') if m else '',
    })

df = df.drop(columns=['label'], errors='ignore').join(df['file'].apply(parse))
bad = df[df['label'] == '']
if len(bad):
    print('⚠️ 이름에 open / half / closed 가 없는 사진 — 아래 계산에서 빠짐'); display(bad[['file']])
df = df[df['label'] != ''].copy()

for c in ['eye_open_prob', 'phone_ear', 'ms', 'ear', 'eye_width_px', *CNN_COLS.values()]:
    df[c] = pd.to_numeric(df.get(c), errors='coerce')
df['face_found'] = df['face_found'].astype(str).str.lower().eq('true')

EXTRA = df[df['cond'] != ''].copy()          # 조건 사진 → 10번
df = df[df['cond'] == ''].copy()             # 기준값은 기본 사진 · 영상 프레임으로만
if len(EXTRA):
    print('조건 사진', len(EXTRA), '장 → 기준값 계산에서 빼고 10번에서 따로'); display(pd.crosstab(EXTRA['cond'], EXTRA['label']))
print('합친 사진', len(df), '장 / CNN:', list(CNN_COLS) or '없음 (CNN 비교 건너뜀)')
display(pd.crosstab(df['person'], df['label'], margins=True))
if (df['light'] != '').any() or (df['source'] != '').any():
    display(pd.crosstab([df['source'], df['light']], df['label'], margins=True))

## 2. 종류별 값 분포

- 상자 그림: 가운데 줄 = 중간값. **open 과 closed 상자가 겹치지 않을수록** 그 방법으로 잘 갈린다
- half(반쯤 감음)가 어느 쪽에 붙는지 본다 — ML Kit 뜸 확률은 half 를 open 쪽으로 보는 경우가 많음 (ML Kit 동작 확인, 예비 시험)
- 값 없음 = ML Kit 이 얼굴 · 눈을 못 찾음 / 눈이 작아 폰 EAR 을 안 씀 (`minEyeWidthPx`) / 서버가 얼굴을 못 찾음

In [ ]:
ORDER = ['open', 'half', 'closed']
VALUES = [('eye_open_prob', 'ML Kit open prob', APP_EYE_CLOSED_PROB),
          ('phone_ear', 'phone EAR (ML Kit contour)', APP_EAR_CLOSED),
          ('ear', 'server EAR (MediaPipe)', SERVER_EAR)]
VALUES += [(col, f'CNN closed prob {i + 1}', CNN_CLOSED_PROB) for i, col in enumerate(CNN_COLS.values())]
print('CNN 번호:', {f'CNN closed prob {i + 1}': name for i, name in enumerate(CNN_COLS)})

rows = []
for col, name, _ in VALUES:
    for lab in ORDER:
        v = df.loc[df['label'] == lab, col]
        rows.append({'값': name, '종류': lab, '장수': len(v), '값 없음': int(v.isna().sum()),
                     '중간값': v.median(), '최소': v.min(), '최대': v.max()})
display(pd.DataFrame(rows).round(3))

fig, axes = plt.subplots(1, len(VALUES), figsize=(4.2 * len(VALUES), 3.6))
for ax, (col, name, thr) in zip(np.atleast_1d(axes), VALUES):
    data = [df.loc[df['label'] == lab, col].dropna() for lab in ORDER]
    ax.boxplot(data)
    ax.set_xticks(range(1, len(ORDER) + 1), ORDER)
    for i, d in enumerate(data, 1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.12, 0.12, len(d)), d, s=10, alpha=0.6)
    ax.axhline(thr, color='red', ls='--', lw=1, label=f'now {thr}')
    ax.set_title(name, fontsize=10); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/2_distribution.png', dpi=90); plt.show()

## 3. 지금 기준값으로 판정

두 가지로 채점한다.

| 채점 | '감음'으로 볼 사진 | 왜 |
|---|---|---|
| **A. 완전히 감음** | closed | 미세수면(1.5초 연속 감김)이 잡아야 하는 것. half 는 빼고 채점 |
| **B. 반쯤 감음 포함** | closed + half | 폰 EAR 을 넣는 이유(반쯤 감긴 졸음). `useContourEar` 를 켤지 정할 때 |

| 칸 | 뜻 |
|---|---|
| 감음 재현율 | 감은 사진을 감음으로 잡은 비율 |
| **놓친 비율** | 감았는데 못 잡은 비율 (= 1 − 감음 재현율, 값 없음 포함) — **가장 중요** |
| 오탐률 | 뜬 사진을 감음으로 본 비율 (괜히 경고) |
| 균형 정확도 | (감음 재현율 + 뜸 재현율) ÷ 2 — 장수가 달라도 공평하게 |
| 값 없음 | 판정을 못 한 사진 수 (앱에서는 '모름' = 감김으로 안 셈 → 감은 사진이면 놓친 것으로 계산) |

In [ ]:
def methods(d, app_prob=APP_EYE_CLOSED_PROB, app_ear=APP_EAR_CLOSED, server_ear=SERVER_EAR, cnn=CNN_CLOSED_PROB):
    # 각 방법의 '감음' 판정 (True / False / NaN = 값 없음)
    def lt(s, t): return s.lt(t).where(s.notna())
    def ge(s, t): return s.ge(t).where(s.notna())
    prob = lt(d['eye_open_prob'], app_prob)
    pear = lt(d['phone_ear'], app_ear)
    # 앱 useContourEar: 뜸 확률이 있으면 '확률 < 기준 또는 EAR < 기준', 확률이 없으면 모름
    both = (prob.fillna(False) | pear.fillna(False)).where(d['eye_open_prob'].notna())
    out = {'ML Kit 뜸 확률': prob, '폰 EAR': pear, 'ML Kit + 폰 EAR': both, '서버 EAR': lt(d['ear'], server_ear)}
    for name, col in CNN_COLS.items():
        out[name] = ge(d[col], cnn)
    return out

def score(pred, truth):
    # pred: 감음 판정(NaN = 값 없음), truth: 정말 감음인가 (bool)
    p = pred.fillna(False).astype(bool)          # 값 없음 = 감음으로 안 잡음
    pos, neg = truth, ~truth
    rec = (p & pos).sum() / max(pos.sum(), 1)
    fa = (p & neg).sum() / max(neg.sum(), 1)
    return {'장수': int(len(truth)), '감음 재현율': rec, '놓친 비율': 1 - rec, '오탐률': fa,
            '균형 정확도': (rec + 1 - fa) / 2, '값 없음': int(pred.isna().sum())}

SCORINGS = {'A. 완전히 감음': (df['label'] != 'half', df['label'] == 'closed'),
            'B. 반쯤 감음 포함': (pd.Series(True, index=df.index), df['label'] != 'open')}

def table(d, scoring_name, **thr):
    keep, truth = SCORINGS[scoring_name][0].loc[d.index], SCORINGS[scoring_name][1].loc[d.index]
    d, truth = d[keep], truth[keep]
    return pd.DataFrame({m: score(p, truth) for m, p in methods(d, **thr).items()}).T

for name in SCORINGS:
    print(f'■ {name} (지금 기준값)')
    display(table(df, name).round(3))

## 4. 기준값 바꿔 보기 + 추천 기준값

- 각 방법의 기준값을 조금씩 바꾸며 **놓친 비율(빨강)** · **오탐률(파랑)** 을 그린다. 세로 점선 = 지금 기준, 초록 = 추천
- 추천 = 놓친 비율 ≤ `MAX_MISS`(5%) 인 기준 중 균형 정확도가 가장 높은 것 (없으면 놓친 비율이 가장 낮은 것)
  - 점수가 같은 기준이 여러 개면 **지금 기준에 가장 가까운 값**
- ⚠️ 이 사진들로 고른 값이라 **출발점**이다. 시연 폰 실제 운행(카메라 영상)으로 한 번 더 확인

In [ ]:
SWEEPS = [('ML Kit 뜸 확률', 'eye_open_prob', 'lt', np.round(np.arange(0.05, 0.96, 0.05), 2), APP_EYE_CLOSED_PROB),
          ('폰 EAR', 'phone_ear', 'lt', np.round(np.arange(0.05, 0.41, 0.01), 2), APP_EAR_CLOSED),
          ('서버 EAR', 'ear', 'lt', np.round(np.arange(0.10, 0.31, 0.01), 2), SERVER_EAR)]
SWEEPS += [(name, col, 'ge', np.round(np.arange(0.1, 0.91, 0.05), 2), CNN_CLOSED_PROB) for name, col in CNN_COLS.items()]

def sweep(d, col, op, grid, scoring_name):
    keep, truth = SCORINGS[scoring_name]
    keep = keep.loc[d.index]
    s = d.loc[keep, col]
    out = []
    for t in grid:
        pred = (s.lt(t) if op == 'lt' else s.ge(t)).where(s.notna())
        out.append({'기준': t, **score(pred, truth.loc[s.index])})
    return pd.DataFrame(out)

def pick(sw, now):
    # 동점이면 지금 기준에 가장 가까운 값 (근거 없이 기준을 크게 옮기지 않게)
    sw = sw.assign(거리=(sw['기준'] - now).abs())
    ok = sw[sw['놓친 비율'] <= MAX_MISS]
    return (ok.sort_values(['균형 정확도', '오탐률', '거리'], ascending=[False, True, True]) if len(ok)
            else sw.sort_values(['놓친 비율', '오탐률', '거리'])).iloc[0]

def recommend(d, scoring_name, plot=None):
    rows = []
    axes = None
    if plot:
        fig, axes = plt.subplots(1, len(SWEEPS), figsize=(4.2 * len(SWEEPS), 3.3))
        axes = np.atleast_1d(axes)
    for i, (name, col, op, grid, now) in enumerate(SWEEPS):
        if d[col].notna().sum() == 0:
            continue
        sw = sweep(d, col, op, grid, scoring_name)
        best = pick(sw, now)
        rows.append({'채점': scoring_name, '방법': name, '지금 기준': now, '추천 기준': best['기준'],
                     '놓친 비율': best['놓친 비율'], '오탐률': best['오탐률'], '균형 정확도': best['균형 정확도']})
        if axes is not None:
            ax = axes[i]
            ax.plot(sw['기준'], sw['놓친 비율'], 'r-', label='miss')
            ax.plot(sw['기준'], sw['오탐률'], 'b-', label='false alarm')
            ax.axvline(now, color='gray', ls='--', lw=1); ax.axvline(best['기준'], color='green', lw=1.5)
            ax.set_title(col, fontsize=10); ax.set_ylim(-0.02, 1.02); ax.legend(fontsize=8)
    if axes is not None:
        plt.suptitle(plot, fontsize=11); plt.tight_layout()
        plt.savefig(f'{OUT_DIR}/4_sweep_{scoring_name[0]}.png', dpi=90); plt.show()
    return pd.DataFrame(rows)

RECOMMEND = pd.concat([recommend(df, n, plot='A: closed only' if n.startswith('A') else 'B: closed + half')
                       for n in SCORINGS], ignore_index=True)
display(RECOMMEND.round(3))

## 5. 서버 확인 단계 — 앱 의심을 확정 / 취소

서버는 앱이 의심해서 보낸 사진을 **한 번 더 보고, '확실히 뜸' 일 때만 취소**한다 (애매하면 경고 쪽, 기획안 5-2).
그래서 서버 방법은 '잘 맞히나' 보다 아래 두 숫자로 고른다.

| 칸 | 뜻 | 좋은 쪽 |
|---|---|---|
| **진짜 졸음 취소** | closed 사진인데 서버가 '확실히 뜸' 으로 취소 → **경고가 사라짐** | 낮을수록 (가장 위험) |
| **반쯤 감음 취소** | half 사진을 서버가 취소 — 폰 EAR(`useContourEar`)이 반쯤 감은 눈을 잡아 보내도 **서버가 취소하면 경고가 안 나감** | `useContourEar` 를 켤 거면 낮아야 함 |
| 오탐 걸러 냄 | open 사진(앱이 잘못 의심했다고 가정)을 서버가 취소 | 높을수록 |
| 얼굴 없음 | 서버가 얼굴을 못 찾음 → 앱 판단대로 확정 (서버 확인이 의미 없음) | 낮을수록 |

- EAR: `EAR ≥ 기준 + CONFIRM_OPEN_MARGIN` 이면 취소 / CNN: `감음 확률 < 기준 − CNN_OPEN_MARGIN` 이면 취소
- 🔴 **서버 규칙 (10/28 CNN 채택 회의):** CNN 이 있으면 **CNN · EAR 둘 다 '확실히 뜸' 일 때만 취소** → `+ EAR (둘 다)` 줄이 실제 서버와 같은 숫자 (`face_service._normalize`)
  - `CNN만` 줄은 비교용 (예전 규칙). 둘 다 뜸이어야 취소하므로 반쯤 감은 눈 취소가 줄고, 실제 주행에서 괜한 경고는 조금 늘 수 있음 (팀 사진은 뜬 눈이 뚜렷해 안 보임 → 시연 폰 시험에서 확인)
- CNN 은 `CNN_PROB_CANDIDATES` (0.4 확정 · 0.5 예전) 를 나란히 비교
- 얼굴이 없으면 CNN 도 못 돌린다 (서버가 EAR → CNN 순서) — 마스크 · 고개 돌림 등
- 10/28 KPI ①: CNN만으로 취소하면 반쯤 감음 취소 약 50% → 'CNN · EAR 둘 다' 규칙으로 바꿈

In [ ]:
def confirm_rates(cancel, d):
    # cancel: 서버가 취소하는가 (NaN = 얼굴 없음 → 취소 안 함)
    c = cancel.fillna(False).astype(bool)
    closed, half, opened = d['label'] == 'closed', d['label'] == 'half', d['label'] == 'open'
    return {'진짜 졸음 취소': c[closed].mean(), '반쯤 감음 취소': c[half].mean() if half.any() else np.nan,
            '오탐 걸러 냄': c[opened].mean(),
            '얼굴 없음': float(cancel.isna().mean()), '장수': int(len(d))}

rows = {f'서버 EAR (기준 {SERVER_EAR}, +{CONFIRM_OPEN_MARGIN})':
        confirm_rates(df['ear'].ge(SERVER_EAR + CONFIRM_OPEN_MARGIN).where(df['ear'].notna()), df)}
for name, col in CNN_COLS.items():
    for t in CNN_PROB_CANDIDATES:
        cnn_open = df[col].lt(t - CNN_OPEN_MARGIN).where(df[col].notna())
        rows[f'{name} (기준 {t}, CNN만 취소 < {t - CNN_OPEN_MARGIN:.2f})'] = confirm_rates(cnn_open, df)
        # 실제 서버 규칙: CNN · EAR 둘 다 확실히 뜸일 때만 취소 (CNN 값이 없으면 얼굴 없음과 같이 NaN)
        ear_open = df['ear'].ge(SERVER_EAR + CONFIRM_OPEN_MARGIN)
        both = (cnn_open.fillna(False).astype(bool) & ear_open).where(df[col].notna())
        rows[f'{name} + EAR (기준 {t}, 둘 다 뜸일 때만 취소)'] = confirm_rates(both, df)
CONFIRM = pd.DataFrame(rows).T
display(CONFIRM.round(3))

## 6. 개인 기준 (출발 때 '정면 5초')

- 사람마다 **뜸 사진 `CALIB_N`(5)장**으로 기준을 만들고 (앱 `calibrate` · 서버 `/drive/calibrate` 와 같은 계산), **그 5장은 평가에서 뺀다** (정답을 미리 본 시험 방지)
  - 앱 뜸 확률: 중간값 × 0.5 → 0.2 ~ 0.4 / 서버 EAR: 중간값 × 0.7 → 0.15 ~ 0.25 / 폰 EAR(제안): 중간값 × 0.7 → 0.10 ~ 0.30
- 고정 기준과 **같은 사진**(5장을 뺀 나머지)으로 비교 → 발표의 '눈이 작은 사람' 답변 근거

In [ ]:
def personal(d):
    calib_idx, th = [], {}
    for person, g in d.groupby('person'):
        o = g[g['label'] == 'open'].sort_values('file').head(CALIB_N)
        calib_idx += list(o.index)
        def med(col, ratio, lo, hi, default):
            v = o[col].dropna()
            return float(np.clip(v.median() * ratio, lo, hi)) if len(v) >= 3 else default   # 3장 미만이면 기본값 (CALIB_MIN_FACES)
        th[person] = {'app_prob': med('eye_open_prob', APP_CALIB_RATIO, APP_CALIB_MIN, APP_CALIB_MAX, APP_EYE_CLOSED_PROB),
                      'phone_ear': med('phone_ear', PHONE_EAR_CALIB_RATIO, PHONE_EAR_CALIB_MIN, PHONE_EAR_CALIB_MAX, APP_EAR_CLOSED),
                      'server_ear': med('ear', SERVER_CALIB_RATIO, SERVER_CALIB_MIN, SERVER_CALIB_MAX, SERVER_EAR)}
    return pd.DataFrame(th).T, calib_idx

PERSONAL, calib_idx = personal(df)
print('사람별 개인 기준'); display(PERSONAL.round(3))
ev = df.drop(index=calib_idx)
print('평가 사진', len(ev), '장 (개인 기준용', len(calib_idx), '장 제외)')

def personal_pred(d):
    t = PERSONAL.loc[d['person']].set_index(d.index)
    lt = lambda s, x: s.lt(x).where(s.notna())
    prob, pear = lt(d['eye_open_prob'], t['app_prob']), lt(d['phone_ear'], t['phone_ear'])
    return {'ML Kit 뜸 확률 (개인)': prob, '폰 EAR (개인)': pear,
            'ML Kit + 폰 EAR (개인)': (prob.fillna(False) | pear.fillna(False)).where(d['eye_open_prob'].notna()),
            '서버 EAR (개인)': lt(d['ear'], t['server_ear'])}

for name, (keep, truth) in SCORINGS.items():
    d = ev[keep.loc[ev.index]]; tr = truth.loc[d.index]
    rows = {m: score(p, tr) for m, p in methods(d).items()}
    rows.update({m: score(p, tr) for m, p in personal_pred(d).items()})
    print(f'■ {name} — 고정 vs 개인 (같은 평가 사진)')
    display(pd.DataFrame(rows).T.round(3))

## 7. 조명별 · 출처별 (영상 / 사진)

- **조명:** 어두운 곳에서 어떤 방법이 얼마나 떨어지는지 → 발표 '한계' 를 숫자로
- **출처:** 영상 프레임(실제 운행과 비슷) vs 사진(고화질). **두 쪽 추천 기준이 비슷하면** "해상도와 상관없이 같은 기준" 으로 말할 수 있고, **다르면 앱 기준은 영상 쪽** 을 따른다

In [ ]:
def by_group(col_name):
    groups = [g for g in df[col_name].unique() if g != '']
    if len(groups) < 2:
        print(f'({col_name} 구분이 없어 건너뜀 — 파일 이름에', 'bright / dark' if col_name == 'light' else '_v번호 / _s번호', '이 없음)')
        return
    keep, truth = SCORINGS['A. 완전히 감음']
    rows = {}
    for g in groups:
        d = df[(df[col_name] == g) & keep]
        rows[g] = {m: score(p, truth.loc[d.index])['균형 정확도'] for m, p in methods(d).items()}
    print(f'■ {col_name} 별 균형 정확도 (채점 A, 지금 기준)'); display(pd.DataFrame(rows).round(3))
    if col_name == 'source':
        rec = pd.concat([recommend(df[df['source'] == g], 'A. 완전히 감음').assign(출처=g) for g in groups])
        print('■ 출처별 추천 기준 (채점 A)')
        display(rec.pivot(index='방법', columns='출처', values='추천 기준'))

by_group('light')
by_group('source')

## 8. 사람별 · 눈 크기 · 폰 EAR vs 서버 EAR

- 사람별 균형 정확도 (채점 A, 지금 기준) — 특정 사람(안경 · 눈이 작은 사람)에서만 떨어지는지
- 눈 폭(px): 작으면 EAR 이 흔들린다. 폰 EAR 이 빈칸인 사진의 눈 폭을 본다 (`minEyeWidthPx` 20)
- 폰 EAR 과 서버 EAR 은 같은 원리지만 점 위치가 달라 **숫자 크기가 다를 수 있다** → 기준값을 따로 정하는 이유

In [ ]:
keep, truth = SCORINGS['A. 완전히 감음']
dA = df[keep]
per = {}
for person, g in dA.groupby('person'):
    per[person] = {m: score(p, truth.loc[g.index])['균형 정확도'] for m, p in methods(g).items()}
print('사람별 균형 정확도 (채점 A, 지금 기준)'); display(pd.DataFrame(per).T.round(3))

print('눈 폭(px, 서버 MediaPipe) — 폰 EAR 있음 / 없음')
display(df.groupby(df['phone_ear'].isna().map({False: '폰 EAR 있음', True: '폰 EAR 없음'}))['eye_width_px'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for lab, c in zip(ORDER, ['tab:green', 'tab:orange', 'tab:red']):
    g = df[df['label'] == lab]
    axes[0].scatter(g['ear'], g['phone_ear'], s=14, c=c, label=lab)
    axes[1].scatter(g['eye_width_px'], g['phone_ear'], s=14, c=c, label=lab)
axes[0].set_xlabel('server EAR'); axes[0].set_ylabel('phone EAR'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('eye width px (server)'); axes[1].set_ylabel('phone EAR')
both = df[['ear', 'phone_ear']].dropna()
if len(both) > 2:
    axes[0].set_title(f'corr = {both.corr().iloc[0, 1]:.2f}', fontsize=10)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/8_phone_vs_server.png', dpi=90); plt.show()
print('ML Kit 처리 시간(ms, 사진 1장 — 원본 크기라 운행 중 카메라 영상보다 느림):', df['ms'].describe().round(0).to_dict())

## 9. 결과 정리 — config 에 넣을 값 · 발표 표

- 요약표를 `OUT_DIR/kpi_eye_summary.csv` · `kpi_eye_recommend.csv` · `kpi_eye_confirm.csv` 로 저장 (드라이브)
- 결정할 것
  1. 앱 뜸 확률 기준 (`config.dart eyeClosedProb`) · 폰 EAR 기준 (`earClosed`)
  2. **`useContourEar` 켜기** — 채점 B 에서 'ML Kit + 폰 EAR' 의 놓친 비율이 'ML Kit 뜸 확률' 보다 확실히 낮고, 채점 A 오탐률이 크게 늘지 않으면 켠다
  3. 서버 EAR 기준 (`config.py EAR_THRESHOLD`)
  4. **CNN 채택 · 모델 · 기준** (3주차 수요일) — 5번 표에서 **진짜 졸음 취소가 EAR 보다 많지 않고**, 3번 채점 A 균형 정확도가 EAR 이상인 CNN → `USE_CNN = True` · `CNN_MODEL_PATH` · `CNN_CLOSED_PROB` (기획안 4-4)

In [ ]:
summary = pd.concat([table(df, n).assign(채점=n) for n in SCORINGS]).reset_index(names='방법')
summary = summary[['채점', '방법', '장수', '놓친 비율', '오탐률', '균형 정확도', '값 없음']]
summary.to_csv(f'{OUT_DIR}/kpi_eye_summary.csv', index=False, encoding='utf-8-sig')
RECOMMEND.to_csv(f'{OUT_DIR}/kpi_eye_recommend.csv', index=False, encoding='utf-8-sig')
CONFIRM.to_csv(f'{OUT_DIR}/kpi_eye_confirm.csv', encoding='utf-8-sig')
display(summary.round(3))

A = RECOMMEND[RECOMMEND['채점'].str.startswith('A')].set_index('방법')['추천 기준']
B = RECOMMEND[RECOMMEND['채점'].str.startswith('B')].set_index('방법')['추천 기준']
print('── config 에 넣을 값 (추천, 사람이 보고 정함) ──')
print(f"config.dart  eyeClosedProb = {A.get('ML Kit 뜸 확률')}   (지금 {APP_EYE_CLOSED_PROB})")
print(f"config.dart  earClosed     = {B.get('폰 EAR')}   (지금 {APP_EAR_CLOSED}, 반쯤 감음 포함 채점으로)")
print(f"config.py    EAR_THRESHOLD = {A.get('서버 EAR')}   (지금 {SERVER_EAR})")

if CNN_COLS:
    sA = summary[summary['채점'].str.startswith('A')].set_index('방법')['균형 정확도']
    ear_cancel = CONFIRM.iloc[0]['진짜 졸음 취소']
    print(f"\n── CNN 채택 (서버 EAR: 균형 정확도 {sA['서버 EAR']:.3f} · 진짜 졸음 취소 {ear_cancel:.3f}) ──")
    for name in CNN_COLS:
        for t in CNN_PROB_CANDIDATES:
            row = CONFIRM.loc[[i for i in CONFIRM.index if i.startswith(name + ' (기준 ' + str(t))][0]]
            ok = sA[name] >= sA['서버 EAR'] and row['진짜 졸음 취소'] <= ear_cancel
            print(f"{name} 기준 {t}: 균형 정확도 {sA[name]:.3f} · 진짜 졸음 취소 {row['진짜 졸음 취소']:.3f} → {'채택 가능' if ok else 'EAR 유지'}")
    print('  (균형 정확도는 지금 기준 CNN_CLOSED_PROB =', CNN_CLOSED_PROB, '으로 계산 — 4번 추천 기준도 함께 볼 것)')
print('저장:', OUT_DIR)

## 10. 조건 사진 — 마스크 · 선글라스 · 고개

| 조건 | 보는 것 | 쓰는 곳 |
|---|---|---|
| 마스크 | 서버 MediaPipe · ML Kit 이 **얼굴을 못 찾는 비율** (B 의 AI-Hub: MediaPipe 47.6% 실패) | 서버가 얼굴을 못 찾으면 앱 판단대로 확정 → 서버 확인이 의미 없어짐 |
| 선글라스 | ML Kit 값 없음 비율 · **뜬 눈을 감음으로 보는 오경보** | "판정 불가" 안내의 근거 |
| 고개 돌림 · 숙임 | 값 없음 · 오경보 · 감은 눈 놓침 | 10/28 결정: 숙임은 CNN 으로 해결 안 됨 → 앱 고개 규칙 (평소보다 6° 이상 숙이면 감김, `config.dart headDownDeg`). 내비를 오래 내려다볼 때 경고는 발표 한계 |

- 장수가 적으니 비율보다 **몇 장 중 몇 장** 으로 본다

In [ ]:
COND = pd.DataFrame()                        # 조건 표 (11번 W&B 기록에서도 씀)
if len(EXTRA) == 0:
    print('조건 사진 없음 (파일 이름에 mask · sun · turn · down 이 없음)')
else:
    rows = {}
    for cond, g in EXTRA.groupby('cond'):
        o, c = g[g['label'] == 'open'], g[g['label'] == 'closed']
        r = {'장수': len(g), 'ML Kit 값 없음': f"{int(g['eye_open_prob'].isna().sum())}/{len(g)}",
             '폰 EAR 없음': f"{int(g['phone_ear'].isna().sum())}/{len(g)}",
             '서버 얼굴 없음': f"{int((~g['face_found']).sum())}/{len(g)}"}
        for m, pred in methods(g).items():
            p = pred.fillna(False).astype(bool)
            r[f'{m} 오경보(뜸→감음)'] = f"{int(p.loc[o.index].sum())}/{len(o)}" if len(o) else '-'
            r[f'{m} 놓침(감음)'] = f"{int((~p.loc[c.index]).sum())}/{len(c)}" if len(c) else '-'
        rows[COND_NAMES.get(cond, cond)] = r
    COND = pd.DataFrame.from_dict(rows, orient='index')     # 칸 순서 그대로
    display(COND)

### 결과 기록 (측정 후 채우기)

| 항목 | 값 |
|---|---|
| 촬영일 · 폰 · 장소 / 조명 | |
| 장수 (사람 × open / half / closed, 영상 / 사진) | |
| 채점 A 놓친 비율 — ML Kit / 서버 EAR / CNN(MRL만) / CNN(미세조정) | |
| 채점 B 놓친 비율 — ML Kit / ML Kit + 폰 EAR | |
| 서버 확인: 진짜 졸음 취소 — EAR / CNN 0.5 / CNN 0.4 | |
| 서버 확인: 반쯤 감음 취소 — EAR / CNN (useContourEar 판단) | |
| 확정 기준값 — eyeClosedProb / earClosed / EAR_THRESHOLD / CNN_CLOSED_PROB | |
| useContourEar · USE_CNN · CNN 모델 결정 | |
| 어두운 곳 균형 정확도 (방법별) | |
| 사진 1장 처리 ms (원본) | |
| 조건 사진 — 마스크 서버 얼굴 없음 / 선글라스 오경보 / 고개 돌림 오경보 | |

- 이 결과는 **사진 · 영상 프레임 파일** 기준이다. 운행 중에는 낮은 해상도의 카메라 영상을 그대로 보므로, 확정 전에 시연 폰 운행 화면에서 한 번 더 확인한다

## 11. W&B 기록 (프로젝트 `eye-kpi1`)

- 강사님 요청 · 팀장 공지 (10/29) — 0~10번을 다 돌린 뒤 실행. 학습이 아니라 '시험' 이라 x 축은 에포크 대신 **기준값**
- run 이름: `a_kpi1_1028_base` (10/28 기본 사진 · 영상) / `a_kpi1_1029_cond` (마스크 · 선글라스 추가)
- 🔴 **올리는 것은 숫자 · 요약 표 · 곡선 · 설정값만** — 사진(`wandb.Image`) · 사진마다 한 줄인 표(`df`) · API 키는 올리지 않음
  - 사람은 a · b · c · d · t 글자로만
- 키는 노트북에 쓰지 않음: PC 는 `wandb login` (한 번), Colab 은 🔑 보안 비밀 `WANDB_API_KEY`
- 인터넷이 안 되면 `WANDB_MODE = 'offline'` → 나중에 `wandb sync wandb/offline-run-*`
- 끝나면 W&B 프로젝트 ⚙️ Settings → **Private** 확인, GitHub 에 올리기 전 출력 지우기

In [ ]:
RUN_NAME = 'a_kpi1_1028_base'      # 10/29 조건 사진은 'a_kpi1_1029_cond'
WANDB_MODE = 'online'              # 인터넷 · 로그인이 안 되면 'offline'

import wandb
try:                                   # Colab 이면 보안 비밀에서 키 (노트북에는 키를 쓰지 않음)
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except ImportError:                    # PC 는 미리 `wandb login` 해 둔 키를 씀
    pass

run = wandb.init(project='eye-kpi1', name=RUN_NAME, group='kpi1', mode=WANDB_MODE,
                 config=dict(app_prob=APP_EYE_CLOSED_PROB, server_ear=SERVER_EAR,
                             confirm_margin=CONFIRM_OPEN_MARGIN, cnn=CNN_CLOSED_PROB,
                             cnn_margin=CNN_OPEN_MARGIN, cnn_models=list(CNN_COLS),
                             n_photos=len(df), n_people=int(df['person'].nunique()),
                             n_cond=len(EXTRA), max_miss=MAX_MISS,
                             server_rule='CNN · EAR 둘 다 확실히 뜸일 때만 취소'),
                 settings=wandb.Settings(console='off'))   # print 가 W&B 로 같이 올라가지 않게

# 1) 요약 표 (숫자만 — 사진마다 한 줄인 df 는 올리지 않음)
keepB, truthB = SCORINGS['B. 반쯤 감음 포함']
perB = {person: {m: score(p, truthB.loc[g.index])['균형 정확도'] for m, p in methods(g).items()}
        for person, g in df[keepB].groupby('person')}
tables = {
    '방법별 결과 (채점 A · B)': summary.round(4),
    '서버 확인 (취소 비율)': CONFIRM.round(4).reset_index(names='방법'),
    '추천 기준값': RECOMMEND.round(4),
    '사람별 균형 정확도 (채점 A)': pd.DataFrame(per).T.round(4).reset_index(names='사람'),
    '사람별 균형 정확도 (채점 B)': pd.DataFrame(perB).T.round(4).reset_index(names='사람'),
}
if len(COND):
    tables['조건 사진 (마스크 · 선글라스 · 고개)'] = COND.reset_index(names='조건').astype(str)
wandb.log({k: wandb.Table(dataframe=v) for k, v in tables.items()})

# 2) 기준값 곡선 — 방법마다 놓친 비율 · 오탐률 (채점 A)
for name, col, op, grid, now in SWEEPS:
    if df[col].notna().sum() == 0:
        continue
    sw = sweep(df, col, op, grid, 'A. 완전히 감음')
    wandb.log({f'기준값 곡선/{name}': wandb.plot.line_series(
        xs=sw['기준'].tolist(), ys=[sw['놓친 비율'].tolist(), sw['오탐률'].tolist()],
        keys=['놓친 비율', '오탐률'], title=f'{name} — 기준값별 (지금 {now})', xname='기준값')})

# 3) 핵심 숫자 (Runs 탭에서 run 끼리 나란히 비교)
for scoring, tag in (('A', 'A'), ('B', 'B')):
    sX = summary[summary['채점'].str.startswith(scoring)].set_index('방법')
    wandb.summary.update({f'균형 정확도 {tag}/{m}': float(v) for m, v in sX['균형 정확도'].items()})
    wandb.summary.update({f'놓친 비율 {tag}/{m}': float(v) for m, v in sX['놓친 비율'].items()})
rule = [i for i in CONFIRM.index if '둘 다' in i and f'기준 {CNN_CLOSED_PROB}' in i]
for i in rule:
    wandb.summary.update({f'서버 확인/{i}/반쯤 감음 취소': float(CONFIRM.loc[i, '반쯤 감음 취소']),
                          f'서버 확인/{i}/진짜 졸음 취소': float(CONFIRM.loc[i, '진짜 졸음 취소'])})
print('W&B run:', run.url)
wandb.finish()